In [ ]:
# 다음리뷰데이터 로드
# 평점 최저 최고 점수 의 절반을 threshold로 정하고 0(긍정) 1(부정)로 추가 컬럼 생성

- 단어를 밀집 백터(Dense Vector)로 변환해서 모델이 단어간의 의미적 유사성을 직접학습
- 단어의 순서 정보를 활용할수 있음
- 학습데이터가 많을수록 단어간의 관계를 더 깊게 파악

- 단점
- 모든문장의 길이를 맞추기위해서 패딩과정이필요 padding
```
- 단어사전관리(Vocabulary) 특수토큰 ( <PAD> ,<UNK> )
```


In [1]:
import pandas as pd
df = pd.read_csv('daum_movie_review.csv')
df.head()

,review,rating,date,title
0,돈 들인건 티가 나지만 보는 내내 하품만,1,2018.10.29,인피니티 워
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,10,2018.10.26,인피니티 워
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,8,2018.10.24,인피니티 워
3,이 정도면 볼만하다고 할 수 있음!,8,2018.10.22,인피니티 워
4,재미있다,10,2018.10.20,인피니티 워


In [ ]:
# 전처리
import re
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
df['cleaned_review'] = df['review'].apply(lambda x : re.sub(r'[^가-힣\s]','',x) )
# 토큰화(한글은 형태소)
okt = Okt()
def kor_tokenize(text):
    return [word for word, pos in okt.pos(text, stem=True) if pos in ['Noun','Verb','Adjective']]

df['data'] = df['cleaned_review'].apply(kor_tokenize)
df.head()

In [10]:
# 단어사전 구축
from collections import Counter
all_tokens = [token for sublist in df['data'].tolist() for token in sublist]
vocab_counts = Counter(all_tokens)
vocab = { word : i + 2  for i, (word,count) in enumerate(vocab_counts.items()) if count >=2}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1
vocab_size = len(vocab)
print(vocab_size)

def text_to_sequence(tokens, vocab):
    return [vocab.get(token, vocab['<UNK>']) for token in tokens]

df['sequence'] = df['data'].apply(lambda x : text_to_sequence(x, vocab))

6774


In [11]:
df.head()

,review,rating,date,title,cleaned_review,data,sequence
0,돈 들인건 티가 나지만 보는 내내 하품만,1,2018.10.29,인피니티 워,돈 들인건 티가 나지만 보는 내내 하품만,"[돈, 들이다, 티, 나, 보다, 내내, 하품]","[2, 3, 4, 5, 6, 7, 8]"
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,10,2018.10.26,인피니티 워,몰입할수밖에 없다 어렵게 생각할 필요없다 내가 전투에 참여한듯 손에 땀이남,"[몰입, 하다, 없다, 어렵다, 생각, 하다, 필요없다, 내, 전투, 참여, 듯, ...","[9, 10, 11, 12, 13, 10, 14, 15, 16, 17, 18, 19..."
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,8,2018.10.24,인피니티 워,이전 작품에 비해 더 화려하고 스케일도 커졌지만 전국 맛집의 음식들을 한데 모은 것...,"[이전, 작품, 비다, 더, 화려하다, 스케일, 커지다, 전국, 맛집, 음식, 모으...","[22, 23, 24, 25, 26, 27, 28, 29, 1, 31, 32, 33..."
3,이 정도면 볼만하다고 할 수 있음!,8,2018.10.22,인피니티 워,이 정도면 볼만하다고 할 수 있음,"[이, 정도, 볼, 하다, 하다, 수, 있다]","[44, 45, 46, 10, 10, 47, 48]"
4,재미있다,10,2018.10.20,인피니티 워,재미있다,[재미있다],[49]


In [13]:
MAX_LEN = 50
def pad_sequence(seq, max_len):
    if len(seq) < max_len:
        return seq + [0]*(max_len - len(seq))
    else:
        return seq[:max_len]

df['padded']  = df['sequence'].apply(lambda x : pad_sequence(x,MAX_LEN))

In [20]:
len(df['padded'][100])

50

In [14]:
from torch.utils.data import Dataset, DataLoader
import torch
class MovieReviewerDataset(Dataset):
    def __init__(self,sequences, labels):
        self.sequences = torch.LongTensor(sequences)
        self.labels = torch.FloatTensor(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return self.sequences[index], self.labels[index]

In [21]:
from sklearn.model_selection import train_test_split
df['target'] =  df['rating'].apply(lambda x : 0 if x > 5 else 1)

x_train,x_test,y_train,y_test = train_test_split(df['padded'].tolist(),df['target'].values,random_state=42,test_size=0.2)
train_loader = DataLoader(MovieReviewerDataset(x_train,y_train), batch_size=32,shuffle=True)
test_loader = DataLoader(MovieReviewerDataset(x_test, y_test), batch_size=32, shuffle=False)

In [22]:
import torch.nn as nn
class EmbeddingMLP(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Embedding(vocab_size,embedding_dim,padding_idx=0),
            nn.Linear(embedding_dim,hidden_dim),
            nn.Dropout(0.5),
            nn.ReLU(),
            nn.Linear(hidden_dim,1)
        )
    def forward(self, x):
        return self.network(x)

In [25]:
import numpy as np
np.array(x_train).shape

(11780, 50)

In [27]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
from torch.optim import Adam
model = EmbeddingMLP(vocab_size, np.array(x_train).shape[1], 64).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), lr=1e-3)

from tqdm import tqdm
epochs = 10
pbar = tqdm(range(epochs), desc="Training")
for epoch in pbar:
    model.train()
    local_loss = 0.0
    for vecs, labels in train_loader:
        vecs, labels = vecs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(vecs).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        local_loss += loss.item()
    avg_loss = local_loss / len(train_loader)
    pbar.set_postfix(
        loss=f"{avg_loss:.4f}"
    )   

Training:   0%|          | 0/10 [00:00<?, ?it/s]


IndexError: index out of range in self